In [1]:
import sys

assert sys.version_info >= (3, 10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [5]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

In [6]:
import deepxde as dde
import numpy as np

torch.manual_seed(42)
torch.cuda.manual_seed(42)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting up the backend

In [7]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")

Set the default float type to float64
Backend: pytorch


Defining Exact solution

In [8]:
def exact_solution(x):
    return (x + 1) ** 2

Defing a domain Geometry with a boudary of -1 and 1

In [9]:
geom = dde.geometry.Interval(-1, 1)


Defining the Partial Differential Equation here second order so hessian

In [10]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx

creating a left boundary condition

In [11]:
def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

Creating a right boundary condition

In [12]:
def boundary_right(x, on_boundary):
    return on_boundary and np.isclose(x[0], 1)

bc_right = dde.icbc.NeumannBC(geom, lambda x: 4, boundary_right)

Generating the data using Numpy

In [13]:
observe_x = np.linspace(-1, 1, 35).reshape(-1, 1)
observe_y  = exact_solution(observe_x)
noise  = 0.1 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)

Combining all the Data

In [14]:
data  = dde.data.PDE(
    geom, pde,
    [bc_left, bc_right, observe],
    num_domain=3000,
    num_boundary=200,
    num_test=500
)

Building a Neural Network

In [16]:
net  = dde.nn.FNN([1, 256, 128, 64, 1], "tanh", "Glorot uniform")
model = dde.Model(data, net)

Creating a class for generating Self Adaptive Weights through a Callback function during training

In [17]:
class SoftmaxAdaptiveWeights(dde.callbacks.Callback):
    def __init__(self, model, every_n_epochs=100, alpha_0=1.0, gamma=0.999):
        super().__init__()
        self.model = model
        self.every_n_epochs = every_n_epochs
        self.alpha = alpha_0
        self.gamma= gamma
        self.epoch =0

    def on_epoch_end(self):
        self.epoch += 1
        if self.epoch % self.every_n_epochs != 0:
            return

        if len(self.model.losshistory.loss_train) ==0:
            return
        current_losses = self.model.losshistory.loss_train[-1]
        exp_terms = np.exp(self.alpha * np.array(current_losses))
        new_weights = exp_terms / np.sum(exp_terms)
        self.alpha *= self.gamma
        self.model.loss_weights = new_weights.tolist()
        print(f"Epoch {self.epoch}: Adaptive weights updated to {new_weights.tolist()}, alpha decayed to {self.alpha:.4f}")

adaptive_callback = SoftmaxAdaptiveWeights(model, every_n_epochs = 20)

Training the MOdel with 2 stage optimization
1. Adam Optimizer
2. L-BFGS

In [18]:
print("\nStage 1: Adam Optimizer")
model.compile("adam", lr = 0.01,
              loss_weights=[1.0] * 4,
              decay=("inverse time", 1000, 0.1))

losshistory, train_state = model.train(iterations=2000, display_every=200, callbacks=[adaptive_callback])
dde.optimizers.config.set_LBFGS_options(maxiter=200)
model.compile("L-BFGS", loss_weights=model.loss_weights)
losshistory, train_state = model.train(display_every=50,callbacks=[adaptive_callback])


Stage 1: Adam Optimizer
Compiling model...
'compile' took 2.465228 s

Training model...



/home/ziaur/ziazh/lib/python3.14/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Step      Train loss                                  Test loss                                   Test metric
0         [2.15e-05, 5.31e-03, 1.66e+01, 3.51e+00]    [1.98e-05, 5.31e-03, 1.66e+01, 3.51e+00]    []  
Epoch 20: Adaptive weights updated to [6.120288810758355e-08, 6.152734190896662e-08, 0.9999978314641849, 2.0458055851710996e-06], alpha decayed to 0.9990
Epoch 40: Adaptive weights updated to [6.222789799209527e-08, 6.255745490080331e-08, 0.9999978024333468, 2.07278130043464e-06], alpha decayed to 0.9980
Epoch 60: Adaptive weights updated to [6.326902360162438e-08, 6.360375832195724e-08, 0.9999977730420158, 2.100085202366174e-06], alpha decayed to 0.9970
Epoch 80: Adaptive weights updated to [6.432650076005028e-08, 6.466648900234102e-08, 0.9999977432860885, 2.1277209218252006e-06], alpha decayed to 0.9960
Epoch 100: Adaptive weights updated to [6.540056846566387e-08, 6.57458869585348e-08, 0.9999977131614192, 2.1556921254197913e-06], alpha decayed to 0.9950
Epoch 120: Adaptive 

Evaluating the results using Metrics

Create the test points using Numpy

In [19]:
x_test =  np.linspace(-1, 1, 15).reshape(-1, 1)
y_exact = exact_solution(x_test)
y_pred = model.predict(x_test)
noise = 0.1 * np.random.randn(15, 1)
y_test = y_exact + noise


Calculating the performance Metrices

In [20]:
absolute_error = np.abs(y_test  - y_pred)
relative_error = absolute_error / (np.abs(y_test) + 1e-10)
l2_error = np.linalg.norm(y_test - y_pred) / np.linalg.norm(y_test)

u_at_minus1 = model.predict(np.array([[-1.0]]))[0, 0]
u_at_plus1 = model.predict(np.array([[1.0]]))[0, 0]

x_right = np.array([[1.0]])
du_dx_at_1 = model.predict(
    x_right,
    operator=lambda x, y: dde.grad.jacobian(y, x, i=0,j=0)
)[0, 0]

print("📊 PERFORMANCE METRICS")
print(f"L2 Relative Error:      {l2_error:.6f}")
print(f"Max Absolute Error:     {np.max(absolute_error):.6f}")
print(f"Mean Absolute Error:    {np.mean(absolute_error):.6f}")

print("\n📍 BOUNDARY CONDITION CHECK")
print(f"u(-1) = {u_at_minus1:.6f}  (Target: 0.0)")
print(f"du/dx(1) = {du_dx_at_1:.6f}  (Target: 4.0)")



📊 PERFORMANCE METRICS
L2 Relative Error:      0.303149
Max Absolute Error:     0.838527
Mean Absolute Error:    0.530823

📍 BOUNDARY CONDITION CHECK
u(-1) = -0.433203  (Target: 0.0)
du/dx(1) = 3.420091  (Target: 4.0)
